In [40]:
# This notebook applies the following basic Machine Learning models:
# Logistic Regression, SVM, KNN and Decision Trees
###

# 0. Preparation
###
# Importing libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from sklearn.metrics import classification_report, f1_score
from sklearn import linear_model, preprocessing
from sklearn.model_selection import train_test_split
from sklearn import svm, neighbors

In [41]:
# Mounting GoogleDrive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [67]:
# Reading data file from GoogleDrive
df = pd.read_pickle("/content/drive/My Drive/Data Science/Team Project X-Rays/Dataframes/df_basic_1.3.pkl")
df.head()

# Define data name
df_name = "Baseline 1.3"

# Define random subsample for computation efficiency
#df = df.sample(500)


In [60]:
# Explore Data
###

# Check data type is DataFrame
print(type(df))

# Show dimensions
print(df.shape)

# Show labels
print(df.Case.value_counts())

# Check distributions after normalisation
df.describe()


<class 'pandas.core.frame.DataFrame'>
(21105, 4098)
Case
Normal             10191
Lung_Opacity        6012
COVID               3564
Viral Pneumonia     1338
Name: count, dtype: int64


,PX_1,PX_2,PX_3,PX_4,PX_5,PX_6,PX_7,PX_8,PX_9,PX_10,...,PX_4087,PX_4088,PX_4089,PX_4090,PX_4091,PX_4092,PX_4093,PX_4094,PX_4095,PX_4096
count,0.0,1.000000,1.000000,1.000000,2.000000,3.000000,3.000000,3.000000,3.000000,5.000000,...,134.000000,131.000000,129.000000,120.000000,118.000000,91.000000,66.000000,44.000000,21.000000,7.000000
mean,NaN,0.505882,0.341176,0.329412,0.343137,0.452288,0.436601,0.452288,0.398693,0.443922,...,0.521335,0.533423,0.520322,0.508235,0.517846,0.544926,0.546999,0.553743,0.554248,0.494678
std,NaN,NaN,NaN,NaN,0.036049,0.221884,0.181426,0.210685,0.224056,0.111037,...,0.163513,0.169909,0.174588,0.172690,0.165027,0.181426,0.184998,0.202622,0.232478,0.149611
min,NaN,0.505882,0.341176,0.329412,0.317647,0.250980,0.239216,0.215686,0.141176,0.254902,...,0.086275,0.000000,0.000000,0.000000,0.137255,0.094118,0.074510,0.066667,0.133333,0.231373
25%,NaN,0.505882,0.341176,0.329412,0.330392,0.333333,0.356863,0.368627,0.323529,0.439216,...,0.403922,0.445098,0.396078,0.407843,0.423529,0.421569,0.445098,0.453922,0.333333,0.435294
50%,NaN,0.505882,0.341176,0.329412,0.343137,0.415686,0.474510,0.521569,0.505882,0.482353,...,0.533333,0.552941,0.545098,0.511765,0.521569,0.556863,0.562745,0.586275,0.584314,0.498039
75%,NaN,0.505882,0.341176,0.329412,0.355882,0.552941,0.535294,0.570588,0.527451,0.517647,...,0.619608,0.649020,0.635294,0.637255,0.639216,0.688235,0.656863,0.704902,0.725490,0.598039
max,NaN,0.505882,0.341176,0.329412,0.368627,0.690196,0.596078,0.619608,0.549020,0.525490,...,0.917647,0.921569,0.945098,0.827451,0.819608,0.929412,0.984314,0.843137,0.937255,0.666667


In [68]:
# Check missing values
print(df.info())

print("Missing vars in columns:\n", df.isna().sum())
print("Number of total missing vars:", df.isna().sum().sum())
print("Number of total missing vars (% of all obs):", (df.isna().sum().sum())/(df.shape[0]*df.shape[1]))

# We have 75% missing vars in total, after masking
# Note: We cannot use basic ML models with NaN != 0. Therefore, dropping 80% NA didnt work.
# Not working for LogisticRegression, SVM and KNN.

# Drop rows if more than X% missing vars per column
#df = df.dropna(thresh = df.shape[1]*0.5, axis = 0)

# Drop columns if more than X% missing vars per column
df = df.dropna(thresh = df.shape[1]*0.8, axis = 0)

# Check missing values
print("---------------")
print(df.info())

print("Missing vars in columns:\n", df.isna().sum())
print("Number of total missing vars:", df.isna().sum().sum())
print("Number of total missing vars (% of all obs):", (df.isna().sum().sum())/(df.shape[0]*df.shape[1]))


# Summary:
# If we drop the rows with more than 80% NA we have no more rows left
# If we drop the columns with more than 80% NA we have no more columns left.
# Moreover, we cannot run basic ML models with NA values present.

# Therefore, dropping NaN doesnt work and we have to interpolate the missing vars.

<class 'pandas.core.frame.DataFrame'>
Index: 21105 entries, 0 to 21164
Columns: 4098 entries, Name to PX_4096
dtypes: float32(4096), object(2)
memory usage: 330.2+ MB
None
Missing vars in columns:
 Name           0
Case           0
PX_1       21105
PX_2       21104
PX_3       21104
           ...  
PX_4092    21014
PX_4093    21039
PX_4094    21061
PX_4095    21084
PX_4096    21098
Length: 4098, dtype: int64
Number of total missing vars: 65100006
Number of total missing vars (% of all obs): 0.7527031231626848
---------------
<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Columns: 4098 entries, Name to PX_4096
dtypes: float32(4096), object(2)
memory usage: 0.0+ bytes
None
Missing vars in columns:
 Name       0
Case       0
PX_1       0
PX_2       0
PX_3       0
          ..
PX_4092    0
PX_4093    0
PX_4094    0
PX_4095    0
PX_4096    0
Length: 4098, dtype: int64
Number of total missing vars: 0
Number of total missing vars (% of all obs): nan


<ipython-input-68-3be0fb376568>:24: RuntimeWarning: invalid value encountered in scalar divide
  print("Number of total missing vars (% of all obs):", (df.isna().sum().sum())/(df.shape[0]*df.shape[1]))


In [64]:
# 1. Data preprocessing
###

# Create categorical variable from Case
df["Case"] = df.Case.replace({"Normal": 0, "COVID": 1, "Lung_Opacity": 2, "Viral Pneumonia": 3})
df.Case.astype(int)

# Check construction
print(df.Case.value_counts())

# Split data into target and features
target = df.Case

# Features data: Drop Names and target
data = df.drop(["Name", "Case"], axis = 1)
data.head()
data.shape

# Split data into training and Test sets, save random state
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size = 0.2, random_state = 123)

Case
0    10191
2     6012
1     3564
3     1338
Name: count, dtype: int64


In [65]:
# 2. Model 1 - Logistic Regression
###
import time
start_time = time.time()

# Instantiate Logistic regression for classification
clf1_name = "Logistic Regression"
clf1 = linear_model.LogisticRegression(solver='lbfgs', C = 1.0, max_iter = 10000)

# Train the model on training data
clf1.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf1.predict(X_test)

# Calc accuracys
clf1_score = clf1.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf1_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model1_time = (time.time() - start_time)/60

ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
# Show Results
###

# Modelling Time
print("Model 1: --- %s minutes ---" % model1_time)

# Score and F1-Score
print("The score is:", clf1_score)
print("The mean F1-Score (unweighted) is:", clf1_f1)

# Show Confusion Matrix
cm1 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
display(cm1)

# Show classification report
model1_cr = classification_report(y_test, y_pred)
print(model1_cr)


In [66]:
# 3. Model 2 - linear SVM
###
import time
start_time = time.time()

# Instantiate SVM
clf2_name = "linear SVM"
clf2 = svm.SVC(gamma = 0.01, kernel = "poly")

# Train the model on training data
clf2.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf2.predict(X_test)

# Calc accuracy
clf2_score = clf2.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf2_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model2_time = (time.time() - start_time)/60

ValueError: Input X contains NaN.
SVC does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [48]:
# Show Results
###

# Modelling Time
print("Model 2: --- %s minutes ---" % model2_time)

# Score and F1-Score
print("The score is:", clf2_score)
print("The mean F1-Score (unweighted) is:", clf2_f1)

# Show Confusion Matrix
cm2 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
display(cm2)

# Show classification report
model2_cr = classification_report(y_test, y_pred)
print(model2_cr)

NameError: name 'model2_time' is not defined

In [49]:
# 3. Model 3 - KNN
###
from sklearn import neighbors
import time
start_time = time.time()

# Instantiate classifier
clf3_name = "KNN"
clf3 = neighbors.KNeighborsClassifier(n_neighbors = 7, metric = 'minkowski')

# Train the model on training data
clf3.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf3.predict(X_test)

# Calc accuracys
clf3_score = clf3.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf3_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model3_time = (time.time() - start_time)/60

ValueError: Input X contains NaN.
KNeighborsClassifier does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [50]:
# Show Results
###

# Modelling Time
print("Model 3: --- %s minutes ---" % model3_time)

# Score and F1-Score
print("The score is:", clf3_score)
print("The mean F1-Score (unweighted) is:", clf3_f1)

# Show Confusion Matrix
cm3 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
display(cm3)

# Show classification report
model3_cr = classification_report(y_test, y_pred)
print(model3_cr)

NameError: name 'model3_time' is not defined

In [51]:
# 3. Model 4 - Decision Tree
###
from sklearn.tree import DecisionTreeClassifier
import time
start_time = time.time()

# Instantiate classifier
clf4_name = "Decision Tree"
clf4 = DecisionTreeClassifier(criterion = "entropy", max_depth = 4, random_state = 123)

# Train the model on training data
clf4.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf4.predict(X_test)

# Calc accuracys
clf4_score = clf4.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf4_f1 = f1_score(y_test, y_pred, average = "weighted")

# Measure time
model4_time = (time.time() - start_time)/60


In [52]:
# Show Results
###

# Modelling Time
print("Model 4: --- %s minutes ---" % model4_time)

# Score and F1-Score
print("The score is:", clf4_score)
print("The mean F1-Score (unweighted) is:", clf4_f1)

# Show Confusion Matrix
cm4 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
display(cm4)

# Show classification report
model4_cr = classification_report(y_test, y_pred)
print(model4_cr)
# Ideas: Could show most important Features here. But well, there are 4000 pixels...

Model 4: --- 0.014098350207010906 minutes ---
The score is: 0.47
The mean F1-Score (unweighted) is: 0.3005442176870748


Predicted Class,0
Realised Class,
0,47
1,16
2,23
3,14


              precision    recall  f1-score   support

           0       0.47      1.00      0.64        47
           1       0.00      0.00      0.00        16
           2       0.00      0.00      0.00        23
           3       0.00      0.00      0.00        14

    accuracy                           0.47       100
   macro avg       0.12      0.25      0.16       100
weighted avg       0.22      0.47      0.30       100



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
